# 20. The July 2026 bulk dataset for Vishal's dashboard

**10 August 2026.**

This notebook builds the one file I still owe Vishal, the bulk table his dashboard joins onto so
the KPI cards stop saying "awaiting file". It is deliberately a build notebook rather than an
analysis one: every decision here was settled beforehand and the reasoning lives in
`drafts/june-2026-bulk-dataset-plan-2026-08-10.md`. What I want from this notebook is that six
months from now, when somebody asks why a column looks the way it does, the answer is on screen
next to the code that produced it.

The filename of that plan still says June. The deliverable is July. Section 1 explains why.

**What ships:** one parquet, one row per company in the July 2026 universe, carrying the Companies
House base columns, the panel deltas, the contract features, the lender features and the four model
scores. Plus a column-mapping document so nobody has to guess what feeds which card.

## 1. The decisions baked into this file

Five things were decided before I opened this notebook, and each one is visible in the code below.

**Base month is July 2026, not June.** This changed late. June was the agreed month out of the
meeting, but Companies House never published a June 2025 snapshot (recorded at
`src/data/ch_bulk.py:31`, and I re-probed the server on 10 August: all fifteen candidate days return
404 while May and July 2025 return 200). Every twelve month lag from a June 2026 origin lands on
that hole, so nine of the twenty five delta features come back 100% NULL at that origin and only
there. LightGBM does not refuse a NaN, it applies the rule it learned during training, which is that
NaN in those columns means "incorporated less than twelve months ago". At a June origin it would
silently apply that reading to all 1.5M companies at once. Forcing the same nine to NaN on the July
frame and rescoring moved 27 of the lending top 100 and 20 of the insolvency top 100. Nothing errors,
which is what makes it dangerous. July looks back to July 2025, which exists, so July is clean.
Vishal confirmed he can crawl the Gazette for July, which removed the only reason June was preferred.

**Scores are reused, not recomputed.** `scores_refactor_growthfix_2026-07.parquet` already exists
from the run that produced the shipped July shortlists. Rescoring would cost about a minute but
would buy nothing and would risk disagreeing with numbers already circulated.

**Contract features come from the May 2026 partition, and are for reading only.** The as-of gate in
`contracts.py` is `publication_date <= last_day(snapshot_date)`, so the July partition needs data
through 31 July. Find a Tender's feed ends 5 June 2026, so both June and July are censored and May
is the last complete month. Censored contract features are not merely noisy, they are biased one
way: they make companies look like they stopped winning work. So I carry May across and flag it.
**The scores were computed from July's own censored contract features and I am deliberately leaving
that inconsistency in place.** Contract features are between 0.2% and 1.0% of any model's SHAP mass,
so refitting against May would change almost nothing and would break agreement with the shortlists.

**Full universe, not the Gazette subset.** Vishal filters to his own matched set. If I pre-filtered,
every change to his filter would mean a new extract from me.

**Lender columns are descriptive only.** The run tag `refactor_growthfix` has `lender_dir=None`, so
no lender feature fed any score. They are here for relationship routing, and keeping that separation
explicit is what stops the leakage argument from ever reaching the dashboard.

In [1]:
import sys
from pathlib import Path

import duckdb

if ".." not in sys.path:
    sys.path.insert(0, "..")

from src.features import charges, contracts, panel
from src.models import targets

BASE_MONTH = "2026-07-01"        # the dashboard month
CONTRACTS_MONTH = "2026-05-01"   # last month inside the harvest watermark
RUN_TAG = "refactor_growthfix"

ROOT = Path("..").resolve()
DELTA_GLOB = f"data/processed/panel_deltas/snapshot_date={BASE_MONTH}/*.parquet"
CONTRACTS_GLOB = f"data/processed/contracts_asof/snapshot_date={CONTRACTS_MONTH}/*.parquet"
LENDER_GLOB = f"data/processed/lender_panel_calib/snapshot_date={BASE_MONTH}/*.parquet"
SCORES_PATH = f"data/processed/scores/scores_{RUN_TAG}_2026-07.parquet"

OUT_DIR = ROOT / "data" / "handover"
OUT_PATH = OUT_DIR / "dashboard_bulk_2026-07.parquet"
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET file_search_path = '{ROOT.as_posix()}'")
print("base month     :", BASE_MONTH)
print("contracts as-of:", CONTRACTS_MONTH)
print("run tag        :", RUN_TAG)
print("output         :", OUT_PATH)

base month     : 2026-07-01
contracts as-of: 2026-05-01
run tag        : refactor_growthfix
output         : /home/viklin/repos/lloyds-commercial-banking-intelligence-2026/data/handover/dashboard_bulk_2026-07.parquet


## 2. The four inputs

Everything already exists on disk. Nothing here is computed from scratch, this notebook is a join.

One choice in that list is not obvious and is worth stating plainly, because picking the wrong one
would ship a known defect. There are six lender panel variants. `data/processed/lender_panel/` is
the module default and it is **the leaking one**: notebook 14b's rewritten leakage test exists
precisely to fail against it, because it gates a charge on `created_on` against the nominal 1st of
the month, and a charge created before `t` is not necessarily knowable by `t`.
`lender_panel_calib/` gates on `visible_on(4d)` and `satisfied_on + 1d` against the real
`source_date`, which is the calibrated point-in-time clock from stream D. **I ship `_calib`.**

In [2]:
for label, src in [
    ("panel_deltas (July)", DELTA_GLOB),
    ("contracts_asof (May)", CONTRACTS_GLOB),
    ("lender_panel_calib (July)", LENDER_GLOB),
    ("scores (July)", SCORES_PATH),
]:
    n_rows, n_cols = con.execute(
        f"SELECT count(*), max(1) FROM read_parquet('{src}')"
    ).fetchone()[0], len(
        con.execute(f"DESCRIBE SELECT * FROM read_parquet('{src}')").fetchall()
    )
    print(f"{label:<28} {n_rows:>10,} rows   {n_cols:>2} cols")

print()
print("contracts harvest watermark:", contracts.harvest_watermark(ROOT / contracts.FLAT_PATH).date())
print("July window ends           : 2026-07-31  (past the watermark, hence the May substitution)")
print("May  window ends           : 2026-05-31  (inside the watermark)")

panel_deltas (July)           1,531,094 rows   43 cols
contracts_asof (May)             44,328 rows    9 cols
lender_panel_calib (July)       158,658 rows   15 cols
scores (July)                 1,409,284 rows    7 cols

contracts harvest watermark: 2026-06-05
July window ends           : 2026-07-31  (past the watermark, hence the May substitution)
May  window ends           : 2026-05-31  (inside the watermark)


## 3. The join

One row per company in the July panel, left joined outwards. Left joins matter in both directions
here: a company with no charges has no lender row, a company that never won public work has no
contract row, and a non-Active company has no score. None of those are errors and all three have to
survive into the file rather than dropping the company.

The sparse blocks coalesce the way the modelling matrix coalesces them, and I take that convention
straight from the same constants `targets._feature_select` uses rather than retyping it. Counts and
flags become 0 or false, because absent means "none". The `months_since_*` columns stay NULL,
because absent means "never", and zero there would read as "this month", which is the opposite.

Two provenance columns go in so the file documents itself rather than depending on a caveats sheet
somebody will lose: `contracts_asof_month` and `contracts_stale`. I kept the six contract features
under their existing names, because the mapping doc and Vishal's code both reference those names and
renaming buys churn rather than clarity.

In [3]:
delta_cols = [r[0] for r in con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{DELTA_GLOB}')"
).fetchall()]

# CH base = whatever the delta panel carries that is not itself a delta feature.
ch_base = [
    c for c in delta_cols
    if c not in panel.DELTA_FEATURE_COLS and c not in ("CompanyNumber", "snapshot_date")
]

lender_extra = ["primary_lender_group"]  # descriptive, not one of the 12 model features


def sparse_select(cols, alias, coalesce_zero, types):
    """Same convention as targets._feature_select: counts to 0, months_since_* left NULL.

    One deviation, on purpose. The modelling SQL writes `coalesce(flag, 0)`, which is
    correct for LightGBM but promotes a BOOLEAN to INTEGER on the way out. That is fine
    inside a feature matrix and confusing in a file somebody opens by hand, so a boolean
    flag coalesces to FALSE and stays boolean. Same values, same meaning, one dtype.
    """
    out = []
    for c in cols:
        if c in coalesce_zero:
            zero = "FALSE" if types[c] == "BOOLEAN" else "0"
            out.append(f'coalesce({alias}."{c}", {zero}) AS "{c}"')
        else:
            out.append(f'{alias}."{c}"')
    return out


def col_types(src):
    return {r[0]: r[1] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{src}')").fetchall()}


select_parts = (
    ['f."CompanyNumber"', f"DATE '{BASE_MONTH}' AS base_month"]
    + [f'f."{c}"' for c in ch_base]
    + [f'f."{c}"' for c in panel.DELTA_FEATURE_COLS]
    + sparse_select(contracts.ASOF_FEATURE_COLS, "c", contracts.ASOF_COALESCE_ZERO,
                    col_types(CONTRACTS_GLOB))
    + [
        f"DATE '{CONTRACTS_MONTH}' AS contracts_asof_month",
        "TRUE AS contracts_stale",
    ]
    + sparse_select(charges.LENDER_FEATURE_COLS, "g", charges.LENDER_COALESCE_ZERO,
                    col_types(LENDER_GLOB))
    + [f'g."{c}"' for c in lender_extra]
    + [
        's."score_lending"',
        's."score_insolvency"',
        's."score_voluntary_exit"',
        's."score_growth"',
    ]
)

BUILD_SQL = f"""
SELECT
    {",\n    ".join(select_parts)}
FROM read_parquet('{DELTA_GLOB}') f
LEFT JOIN read_parquet('{CONTRACTS_GLOB}') c USING ("CompanyNumber")
LEFT JOIN read_parquet('{LENDER_GLOB}')    g USING ("CompanyNumber")
LEFT JOIN read_parquet('{SCORES_PATH}')    s USING ("CompanyNumber")
"""

print(f"{len(select_parts)} columns")
print(f"  identity + base month  : 2")
print(f"  CH base                : {len(ch_base)}")
print(f"  panel deltas           : {len(panel.DELTA_FEATURE_COLS)}")
print(f"  contracts (+ 2 flags)  : {len(contracts.ASOF_FEATURE_COLS)} + 2")
print(f"  lender (+ group)       : {len(charges.LENDER_FEATURE_COLS)} + {len(lender_extra)}")
print(f"  scores                 : 4")

69 columns
  identity + base month  : 2
  CH base                : 16
  panel deltas           : 25
  contracts (+ 2 flags)  : 7 + 2
  lender (+ group)       : 12 + 1
  scores                 : 4


In [4]:
con.execute(f"CREATE OR REPLACE TEMP TABLE bulk AS {BUILD_SQL}")
print(f"{con.execute('SELECT count(*) FROM bulk').fetchone()[0]:,} rows built")

1,531,094 rows built


## 4. Verification

Six checks. The first four are the ones that would catch a broken join, the last two are the ones
that would catch me shipping the wrong month, which given how this week has gone is the failure I am
more worried about.

### 4.1 Shape and keys

In [5]:
shape = con.execute("""
    SELECT count(*)                        AS rows_,
           count(DISTINCT "CompanyNumber")  AS distinct_companies,
           count(*) FILTER (WHERE "CompanyNumber" IS NULL) AS null_keys
    FROM bulk
""").df()
print(shape.to_string(index=False))

src_rows = con.execute(f"SELECT count(*) FROM read_parquet('{DELTA_GLOB}')").fetchone()[0]
assert shape.loc[0, "rows_"] == src_rows, "join changed the row count"
assert shape.loc[0, "rows_"] == shape.loc[0, "distinct_companies"], "duplicate company numbers"
assert shape.loc[0, "null_keys"] == 0
print("\nOK: one row per company, no fan-out, no null keys.")

  rows_  distinct_companies  null_keys
1531094             1531094          0

OK: one row per company, no fan-out, no null keys.


### 4.2 Score coverage lines up with `is_active`

The scoring path filters on `is_active`, so scores must be present on exactly the active rows and
absent everywhere else. If those two sets disagree by even one company, something is wrong with
either the join or my understanding of the scoring stage.

In [6]:
cov = con.execute("""
    SELECT count(*) FILTER (WHERE is_active)                          AS active,
           count(*) FILTER (WHERE score_lending IS NOT NULL)          AS scored,
           count(*) FILTER (WHERE is_active AND score_lending IS NULL) AS active_unscored,
           count(*) FILTER (WHERE NOT is_active AND score_lending IS NOT NULL) AS inactive_scored,
           round(100.0 * count(*) FILTER (WHERE score_lending IS NULL) / count(*), 2) AS pct_unscored
    FROM bulk
""").df()
print(cov.to_string(index=False))
assert cov.loc[0, "active_unscored"] == 0 and cov.loc[0, "inactive_scored"] == 0
print("\nOK: scored set is exactly the active set. The unscored share is caveat D1.")

 active  scored  active_unscored  inactive_scored  pct_unscored
1409284 1409284                0                0          7.96

OK: scored set is exactly the active set. The unscored share is caveat D1.


### 4.3 The sparse blocks coalesced the way they should

Counts at zero for companies with no row, `months_since_*` still NULL for them.

In [7]:
sparse = con.execute("""
    SELECT
      count(*) FILTER (WHERE ever_won_contract)                  AS has_contract_history,
      count(*) FILTER (WHERE contracts_won_12m IS NULL)          AS contracts_null_count,
      count(*) FILTER (WHERE months_since_last_award IS NULL)    AS never_won_award,
      count(*) FILTER (WHERE n_charges_outstanding > 0)          AS has_outstanding_charge,
      count(*) FILTER (WHERE n_charges_outstanding IS NULL)      AS lender_null_count,
      count(*) FILTER (WHERE is_lbg_client)                      AS lbg_clients,
      count(*) FILTER (WHERE ever_lbg_client AND NOT is_lbg_client) AS lapsed_lbg
    FROM bulk
""").df()
print(sparse.to_string(index=False))
assert sparse.loc[0, "contracts_null_count"] == 0, "contract counts should coalesce to 0"
assert sparse.loc[0, "lender_null_count"] == 0, "lender counts should coalesce to 0"
print("\nOK: counts coalesced to 0, months_since_* left NULL for 'never'.")

 has_contract_history  contracts_null_count  never_won_award  has_outstanding_charge  lender_null_count  lbg_clients  lapsed_lbg
                15517                     0          1515577                  103539                  0        11733       14416

OK: counts coalesced to 0, months_since_* left NULL for 'never'.


### 4.4 Vishal's universe is recoverable with one predicate

This is the reconciliation from wave 1. His widened universe and mine differ by companies whose SIC
in the current month falls outside our target sectors but which were in-sector in another month. My
panel keeps them, deliberately, so that a SIC recode does not look like a dissolution. So he can
recover his own population from my file with a `WHERE`, and does not need a second extract.

In [8]:
recon = con.execute("""
    SELECT count(*)                                                        AS full_universe,
           count(*) FILTER (WHERE sector IS NOT NULL)                      AS his_widened_universe,
           count(*) FILTER (WHERE sector IS NOT NULL AND is_active)        AS his_trading_bucket,
           count(*) FILTER (WHERE sector IS NULL)                          AS sector_null
    FROM bulk
""").df()
print(recon.to_string(index=False))
print("\nFor June these predicates reproduced his 1,493,972 and 1,372,321 exactly.")
print("These are the July equivalents he should expect after rebuilding.")

 full_universe  his_widened_universe  his_trading_bucket  sector_null
       1531094               1505203             1385001        25891

For June these predicates reproduced his 1,493,972 and 1,372,321 exactly.
These are the July equivalents he should expect after rebuilding.


### 4.5 The nine twelve-month features are alive

This is the check that the whole base-month change exists for. At a June 2026 origin every one of
these is 100% NULL. If any of them comes back at zero here, I have built the wrong month and the
scores in this file are describing a population the model has misread.

In [9]:
nine = [
    "d_charges_12m", "d_outstanding_12m", "d_satisfied_12m", "debt_ratio_trend_12m",
    "segment_upgraded_12m", "segment_downgraded_12m",
    "sic_changed_12m", "name_changed_12m", "postcode_changed_12m",
]
sel = ",\n           ".join(
    f'round(avg(CASE WHEN "{c}" IS NOT NULL THEN 1.0 ELSE 0.0 END), 4) AS "{c}"' for c in nine
)
alive = con.execute(f"SELECT {sel} FROM bulk WHERE is_active").df().T
alive.columns = ["non_null_rate"]
print(alive.to_string())
assert (alive["non_null_rate"] > 0.5).all(), "a 12m feature is dead: wrong base month"
print("\nOK: all nine populated. This is what June 2026 could not give us.")

                        non_null_rate
d_charges_12m                  0.8505
d_outstanding_12m              0.8505
d_satisfied_12m                0.8505
debt_ratio_trend_12m           0.8505
segment_upgraded_12m           0.5817
segment_downgraded_12m         0.5817
sic_changed_12m                0.8505
name_changed_12m               0.8505
postcode_changed_12m           0.8505

OK: all nine populated. This is what June 2026 could not give us.


### 4.6 The contract block really is May, and the scores really are July

The one deliberate inconsistency in the file. I want it asserted rather than assumed, so that if
somebody later "fixes" it by swapping in the July contract partition, this cell fails and tells them
why that was on purpose.

In [10]:
prov = con.execute("""
    SELECT DISTINCT base_month, contracts_asof_month, contracts_stale FROM bulk
""").df()
print(prov.to_string(index=False))

# The scores were produced from July's own contract features, not from these columns.
origin = con.execute(f"SELECT DISTINCT origin_month FROM read_parquet('{SCORES_PATH}')").df()
print("\nscore origin month:", origin.iloc[0, 0])
print("contract columns  :", prov.loc[0, "contracts_asof_month"], "(EDA only, did not feed scores)")
assert str(prov.loc[0, "contracts_asof_month"].date()) == CONTRACTS_MONTH
assert str(origin.iloc[0, 0].date()) == BASE_MONTH
print("\nOK: contracts are May by design, scores are July. Do not 'fix' this, see section 1.")

base_month contracts_asof_month  contracts_stale
2026-07-01           2026-05-01             True

score origin month: 2026-07-01 00:00:00
contract columns  : 2026-05-01 00:00:00 (EDA only, did not feed scores)

OK: contracts are May by design, scores are July. Do not 'fix' this, see section 1.


## 5. Write it out

zstd rather than snappy. It came out about a third smaller when I measured both, and at this size
the difference is the gap between a file that goes in a message and one that needs somewhere to live.

In [11]:
con.execute(
    f"COPY bulk TO '{OUT_PATH.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
size_mb = OUT_PATH.stat().st_size / 1e6
n_rows, n_cols = con.execute(
    f"SELECT count(*) FROM read_parquet('{OUT_PATH.as_posix()}')"
).fetchone()[0], len(
    con.execute(f"DESCRIBE SELECT * FROM read_parquet('{OUT_PATH.as_posix()}')").fetchall()
)
print(f"wrote {OUT_PATH}")
print(f"{n_rows:,} rows x {n_cols} columns, {size_mb:.1f} MB, {size_mb * 1e6 / n_rows:.1f} bytes/row")

wrote /home/viklin/repos/lloyds-commercial-banking-intelligence-2026/data/handover/dashboard_bulk_2026-07.parquet
1,531,094 rows x 69 columns, 64.0 MB, 41.8 bytes/row


In [12]:
# Read it back cold, as Vishal will, and print the full schema for the record.
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{OUT_PATH.as_posix()}')"
).df()[["column_name", "column_type"]]
print(schema.to_string())

                             column_name column_type
0                          CompanyNumber     VARCHAR
1                             base_month        DATE
2                            source_date   TIMESTAMP
3               Mortgages.NumMortCharges      BIGINT
4           Mortgages.NumMortOutstanding      BIGINT
5             Mortgages.NumMortSatisfied      BIGINT
6                          CompanyStatus     VARCHAR
7                              is_active     BOOLEAN
8                                 sector     VARCHAR
9                                segment     VARCHAR
10                             size_tier     VARCHAR
11                             tier_rank     INTEGER
12                     SICCode.SicText_1     VARCHAR
13                           CompanyName     VARCHAR
14                   RegAddress.PostCode     VARCHAR
15                     company_age_years      DOUBLE
16                            debt_ratio      DOUBLE
17                      accounts_overdue     B

## 6. What goes with the file

The parquet is not the whole handover. Two documents ship beside it:

- `reports/dashboard_handover_columns.md`, the column mapping, which says what every column means
  and which KPI card it feeds, and marks each card satisfiable, partly satisfiable or not
  satisfiable from what we have.
- The caveats in section 7 of `drafts/june-2026-bulk-dataset-plan-2026-08-10.md`, which are written
  to be pasted to Vishal as they stand.

The three caveats I would not let him miss, in order of how much damage they do if missed:

**Non-Active companies have no scores, and that is correct.** The models are fitted on trading
companies. Because his population is Gazette-matched, and a company with a winding-up petition is on
its way out of Active status, a large share of the companies on his screen will have empty score
cards. A lending-readiness forecast for a company already in liquidation would be a number with
nothing behind it. The card should say so rather than render a zero.

**The scores are forecasts about August 2026 onwards, not descriptions of July.** Each one needs its
horizon on screen next to it, and the wording should lead with the measured hit rate rather than the
raw probability, which runs optimistic at the top of the list where the dashboard looks.

**Contract activity is as at 31 May 2026.** Find a Tender's feed ends 5 June, so June and July
coverage is partial and would understate recent awards. The file says so in its own columns.

### Still open

Per-company score drivers do not exist. Only the global `shap_importance_*.csv` does, about forty
rows per target, which is a statement about the model rather than about a company. Building the
per-company version is cheap via LightGBM's native `pred_contrib=True`, and it is the single most
valuable thing to add after this file, because it turns a bare number into a call script. It is
deliberately out of scope here so that the blocking handover ships first.